In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()

import numpy as np
import pandas as pd

In [2]:
api_key = os.getenv("OPENAI_API_KEY")

In [3]:
df = pd.read_csv("./laptops.csv")
df.head()

,laptop_description
0,Lenovo IdeaPad 1: This is a budget-friendly la...
1,Lenovo IdeaPad 3: This laptop is designed as a...
2,Lenovo IdeaPad 5: This is a versatile laptop a...
3,Lenovo IdeaPad Slim: This laptop emphasizes a ...
4,Lenovo IdeaPad 2-in-1: This convertible laptop...


In [4]:
#!pip install tenacity
from tenacity import retry, wait_random_exponential, stop_after_attempt

client = OpenAI(api_key=api_key)

MODEL = 'gpt-3.5-turbo-16k'

SYSTEM_PROMPT = """
You are a laptop classification assistant.

Classify the laptop description into exactly ONE of these categories:

- general: Everyday use, browsing, documents, study, basic tasks
- business: Office work, portability, battery life, professional use
- gamer: Gaming, dedicated GPU, high performance, graphics-intensive tasks
- programmer: Programming, development, coding, high RAM, strong CPU
- multimedia: Movies, music, high-resolution display, speakers, content consumption

Return ONLY the category name.

Valid outputs:
general
business
gamer
programmer
multimedia
"""

In [8]:
@retry(
    wait=wait_random_exponential(min=1, max=20),
    stop=stop_after_attempt(6)
)
def get_response(description):
    
    try:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Laptop description:\n{description}"}
        ]

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages
        )

        return response.choices[0].message.content.strip().lower()
    
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [9]:
def tag_laptops(df):
    a = df.copy()

    a["category"] = [
        get_response(description)
        for description in a["laptop_description"]
    ]

    return a

In [10]:
tag_df = tag_laptops(df)
tag_df

,laptop_description,category
0,Lenovo IdeaPad 1: This is a budget-friendly la...,general
1,Lenovo IdeaPad 3: This laptop is designed as a...,general
2,Lenovo IdeaPad 5: This is a versatile laptop a...,general
3,Lenovo IdeaPad Slim: This laptop emphasizes a ...,business
4,Lenovo IdeaPad 2-in-1: This convertible laptop...,business
5,HP Pavilion: This laptop is designed for a bro...,general
6,HP Pavilion Aero: This is a lightweight laptop...,business
7,HP Pavilion Plus: This laptop targets users lo...,multimedia
8,HP Pavilion x360: This convertible laptop comb...,multimedia
9,Dell Inspiron 14 2-in-1: This laptop is design...,general
